In [1]:
import sys 
sys.path.append('/home/hydrogen/workspace/Space_GW/pespace')
sys.path.append('/home/hydrogen/workspace/Space_GW/wf4ti')

import taichi as ti
ti.init()
import numpy as np
import h5py
from matplotlib import pyplot as plt
%matplotlib inline

[Taichi] version 1.6.0, llvm 15.0.4, commit f1c6fbbd, linux, python 3.10.12


[I 07/25/24 10:45:28.291 1900656] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


[Taichi] Starting on arch=x64


In [ ]:
# file_path = "/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v2_TD_9gc2s16.hdf5"
file_path = "/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v1_1_TD.hdf5"
with h5py.File(file_path, "r") as f:
    # f.visit(lambda name: print(name))

    source_parameters = {}
    print('Source information: ')
    for parameter, value in f["H5LISA/GWSources/MBHB-0"].items():
        print(f"{parameter}: {value[()]}")
        source_parameters[parameter] = value[()]

    print(f["H5LISA/PreProcess/TDIGenerator"][()])
    TDI_data = f["H5LISA/PreProcess/TDIdata"][()]

In [ ]:
time_samples = TDI_data[:,0]
TDI_strain = TDI_data[:,1:]
print(TDI_strain)
print(TDI_strain.shape)
TDI_strain = TDI_strain.T
print(TDI_strain)
print(TDI_strain.shape)

duration = time_samples[-1]-time_samples[0]
cadence = time_samples[1]-time_samples[0]
print('duration: ', duration)
print('cadence: ', cadence)
print(source_parameters['ObservationDuration'])
print(source_parameters['Cadence'])

In [ ]:
from pespace.detectors import TDIChannelsData


In [ ]:
LDC_mbhb = TDIChannelsData()
LDC_mbhb.set_data_info(channels=('X', 'Y', 'Z'), 
                                               generation='1.5', 
                                               duration=duration,
                                               cadence=cadence,
                                               start_time=time_samples[0]
                                               )
LDC_mbhb.data_info
LDC_mbhb.set_time_domain_data_from_input_array(channels=('X', 'Y', 'Z'), 
                                               generation='1.5', 
                                               duration=duration,
                                               cadence=cadence,
                                               TDI_data_array=TDI_strain,
                                               start_time=time_samples[0])
print(LDC_mbhb.time_domain_TDI_data)
print(LDC_mbhb.time_samples)
# check the recovered array
print(np.sum(LDC_mbhb.time_samples.to_numpy()-time_samples))
for idx, chan in enumerate(('X', 'Y', 'Z')):
    print(np.sum(LDC_mbhb.time_domain_TDI_data.get_member_field(chan).to_numpy() - TDI_strain[idx]))

plt.figure()
for chan in ('X', 'Y', 'Z'):
    plt.plot(LDC_mbhb.time_samples_numpy_array, LDC_mbhb.time_domain_TDI_data_numpy_array[chan], label=chan)
plt.legend()



In [ ]:
LDC_mbhb.Fourier_transform_time_domain_data_to_frequency_domain()
print(LDC_mbhb.frequency_domain_TDI_data_numpy_array)

In [ ]:
plt.figure()
for chan in ('X', 'Y', 'Z'):
    plt.loglog(LDC_mbhb.frequency_samples_numpy_array, np.abs(LDC_mbhb.frequency_domain_TDI_data_numpy_array[chan]), label=chan)
plt.legend()


In [ ]:
from numpy.typing import NDArray

fd_data = LDC_mbhb.frequency_domain_TDI_data
print(fd_data.shape==(LDC_mbhb.data_info.frequency_series_length,))

In [ ]:
print(fd_data.__dir__())
['field_dict', 'struct_methods', 'name', 'grad', 'dual', 'X', 'Y', 'Z', '__module__', '__doc__', '__init__', 'keys', '_members', '_items', '_make_getter', '_make_setter', '_register_fields', '_get_field_members', '_snode', '_loop_range', 'copy_from', 'fill', '_initialize_host_accessors', 'get_member_field', 'from_numpy', 'from_torch', 'from_paddle', 'to_numpy', 'to_torch', 'to_paddle', '__setitem__', '__getitem__', 'snode', 'shape', 'dtype', '_name', 'parent', '_set_grad', '_set_dual', '_from_external_arr', '__str__', '_pad_key', '_host_access', '__iter__', '__dict__', '__weakref__', '__new__', '__repr__', '__hash__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__eq__', '__ne__', '__gt__', '__ge__', '__reduce_ex__', '__reduce__', '__subclasshook__', '__init_subclass__', '__format__', '__sizeof__', '__dir__', '__class__']

In [ ]:
vec2_c = ti.types.vector(2, ti.f64)
d = fd_data.X
@ti.kernel
def test():
    for i in d:
        # d[i] += vec2_c([0.0, 0.2])
        print(d[i])
test()

In [ ]:
# DataInfo(channels=('X', 'Y', 'Z'), 
#          generation='1.5', duration=41943030.0, cadence=10.0, start_time=10.0, fmin_in=1e-05, fmax_in=0.1, minimum_frequency=1e-05, maximum_frequency=0.05, sampling_frequency=0.1, delta_frequency=2.3841863594499493e-08, 
         
#          time_series_length=4194304, 
#          time_samples_array=array([1.000000e+01, 2.000000e+01, 3.000000e+01, ..., 4.194302e+07,
#        4.194303e+07, 4.194304e+07]), 
#        full_frequency_series_length=2097153, 
#        full_frequency_samples_array=array([0.00000000e+00, 2.38418636e-08, 4.76837272e-08, ...,
#        4.99999642e-02, 4.99999881e-02, 5.00000119e-02]), frequency_mask_array=array([False, False, False, ...,  True,  True, False]), 
#        frequency_samples_array=array([1.00135827e-05, 1.00374246e-05, 1.00612664e-05, ...,
#        4.99999404e-02, 4.99999642e-02, 4.99999881e-02]), 
#        frequency_series_length=2096732
#        )

In [ ]:
n = 10
particle_field = ti.Struct.field({
    "pos": ti.math.vec3,
    "vel": ti.math.vec3,
    "acc": ti.math.vec3,
    "mass": float,
  }, shape=(n,))
print(isinstance(particle_field, ti.StructField))


In [ ]:

a = ti.field(dtype=ti.types.vector(2, ti.f64), shape=(10,))
b = ti.field(dtype=ti.types.vector(2, ti.f64), shape=(10,))
@ti.kernel
def test():
    for i in a:
        a[i]= ti.math.vec2([0, 0.2])
        print(a[i])
    for i in b:
        b[i]= ti.math.vec2([1, 0])
        print(b[i])
    for i in a:
        a[i]+= b[i]
        print(a[i])
    for i in a:
        print(a[i])
test()

In [37]:
a=[1,2,3,4,5,6,7,8,9]
a_f = np.fft.rfft(a)
print(a_f)
af = [45.  +0.j   ,      -4.5+12.36364839j ,-4.5 +5.36289117j ,-4.5 +2.59807621j,-4.5 +0.79347141j]
a_t = np.fft.irfft(a_f, )
print(a_t)

[45.  +0.j         -4.5+12.36364839j -4.5 +5.36289117j -4.5 +2.59807621j
 -4.5 +0.79347141j]
[ 1.6875      2.20189298  3.74610696  4.88333856  6.1875      7.49166144
  8.62889304 10.17310702]


In [49]:
a = np.array([[1,2],[3,4],[5,6]])
b= np.array([10, 100, 100]).T
print(a)
print(b)

[[1 2]
 [3 4]
 [5 6]]
[ 10 100 100]


In [51]:
a*(b.T)

ValueError: operands could not be broadcast together with shapes (3,2) (3,) 

In [15]:
a = ti.Struct.field({'a':ti.math.vec2, 'b':ti.math.vec2, }, shape=(10))
b =a.to_numpy()

In [20]:
print([len(data)==10 for _, data in b.items()])

[True, True]


In [13]:
mat3= ti.types.matrix(3,3, dtype=ti.f64)
vec3 = ti.types.vector(3, dtype=ti.f64)
h = mat3(1,2,3,4,5,6,7,8,9)
n1 = vec3(1,2,3)
n2 = vec3(1,2,3)
print(h)
print(n1)
print(n2)
print(h@n1)
print(n1@h)

[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
[1. 2. 3.]
[1. 2. 3.]
[14. 32. 50.]
[30. 36. 42.]


In [1]:
a= dict(a=1, b=2)

In [3]:
a.keys()

dict_keys(['a', 'b'])

In [1]:
import taichi as ti
ti.init(offline_cache=False)

@ti.dataclass
class STest:
    a: ti.i8
    b: ti.i8

    def set_init(self, a, b):
        self.a = a
        self.b = b

vec = STest.field(shape=())
vec[None].a = 1
vec2 = STest()

@ti.kernel
def my_kernel(x: ti.template()):
    print(x[None].a)

print(vec)
my_kernel(vec)
vec[None].a = 10
print(vec)
my_kernel(vec)

[Taichi] version 1.7.2, llvm 15.0.4, commit 0131dce9, linux, python 3.10.14


[I 08/31/24 18:51:04.884 3029343] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


[Taichi] Starting on arch=x64
{'a': array(1, dtype=int8), 'b': array(0, dtype=int8)}
{'a': array(10, dtype=int8), 'b': array(0, dtype=int8)}
1
10


In [1]:
import taichi as ti

ti.init(arch=ti.gpu, offline_cache=False)
init = ti.field(ti.i8, shape=(2, 3))
init[0, 0] = 1

[Taichi] version 1.7.2, llvm 15.0.4, commit 0131dce9, linux, python 3.10.14


[I 08/31/24 18:55:17.519 3029928] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


[Taichi] Starting on arch=cuda
